In [2]:
"""
Car Price Prediction with Machine Learning
=============================================
Predicts the selling price of used cars from features such as brand/model,
age, mileage (kms driven), present (new) price, fuel type, seller type,
transmission and ownership history.

Pipeline:
    1. Load & explore data
    2. Clean / preprocess data
    3. Engineer features (car age, km per year, target-encoded brand/model)
    4. Build preprocessing + model pipelines
    5. Compare several regression models with cross-validation
    6. Tune the best model with GridSearchCV
    7. Evaluate on a held-out test set (R2, MAE, RMSE, MAPE)
    8. Visualize results (saved as PNG files) and feature importance
    9. Persist the final trained model to disk

Dataset columns (car_data.csv):
    Car_Name, Year, Selling_Price, Present_Price, Driven_kms,
    Fuel_Type, Selling_type, Transmission, Owner

Usage:
    Run all cells in Jupyter, or: python car_price_prediction.py [path/to/car_data.csv]
"""

import os
import warnings
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---------------------------------------------------------------------------
# DATA PATH  ← edit this if your CSV lives elsewhere
# ---------------------------------------------------------------------------
DATA_PATH = "cardata.csv"


# ---------------------------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------------------------
def load_data(path: str) -> pd.DataFrame:
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"CSV not found at '{path}'.\n"
            "Set the DATA_PATH variable at the top of this script to the correct location."
        )
    df = pd.read_csv(path)
    print(f"Loaded data: {df.shape[0]} rows, {df.shape[1]} columns")
    return df


# ---------------------------------------------------------------------------
# 2. EXPLORATORY DATA ANALYSIS
# ---------------------------------------------------------------------------
def explore_data(df: pd.DataFrame) -> None:
    print("\n--- Data types & missing values ---")
    print(df.dtypes)
    print("\nMissing values per column:")
    print(df.isna().sum())

    print("\n--- Summary statistics ---")
    print(df.describe(include="all").T)

    print(f"\nDuplicate rows: {df.duplicated().sum()}")
    print(f"Unique Car_Name values: {df['Car_Name'].nunique()}")

    # Visualizations
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))

    axes[0, 0].hist(df["Selling_Price"], bins=30, color="steelblue", edgecolor="white")
    axes[0, 0].set_title("Distribution of Selling Price (lakhs)")
    axes[0, 0].set_xlabel("Selling Price")

    axes[0, 1].scatter(df["Present_Price"], df["Selling_Price"], alpha=0.5, color="darkorange")
    axes[0, 1].set_title("Present Price vs Selling Price")
    axes[0, 1].set_xlabel("Present Price")
    axes[0, 1].set_ylabel("Selling Price")

    axes[1, 0].scatter(df["Driven_kms"], df["Selling_Price"], alpha=0.5, color="seagreen")
    axes[1, 0].set_title("Kms Driven vs Selling Price")
    axes[1, 0].set_xlabel("Driven Kms")
    axes[1, 0].set_ylabel("Selling Price")

    df.boxplot(column="Selling_Price", by="Fuel_Type", ax=axes[1, 1])
    axes[1, 1].set_title("Selling Price by Fuel Type")
    plt.suptitle("")

    plt.tight_layout()
    plt.savefig("eda_overview.png", dpi=150)
    plt.close()
    print("\nSaved EDA plots to eda_overview.png")

    # Correlation heatmap for numeric columns
    numeric_cols = df.select_dtypes(include=[np.number])
    corr = numeric_cols.corr()
    plt.figure(figsize=(6, 5))
    plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
    plt.yticks(range(len(corr.columns)), corr.columns)
    for i in range(len(corr.columns)):
        for j in range(len(corr.columns)):
            plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.savefig("correlation_heatmap.png", dpi=150)
    plt.close()
    print("Saved correlation heatmap to correlation_heatmap.png")


# ---------------------------------------------------------------------------
# 3. FEATURE ENGINEERING
# ---------------------------------------------------------------------------
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    current_year = datetime.now().year

    df["Car_Age"] = current_year - df["Year"]
    df["Km_Per_Year"] = df["Driven_kms"] / df["Car_Age"].replace(0, 1)
    df["Price_Ratio"] = df["Present_Price"] / (df["Car_Age"] + 1)

    df = df.drop(columns=["Year"])
    return df


# ---------------------------------------------------------------------------
# 4. PREPROCESSING PIPELINE
# ---------------------------------------------------------------------------
def build_preprocessor() -> ColumnTransformer:
    numeric_features = [
        "Present_Price",
        "Driven_kms",
        "Owner",
        "Car_Age",
        "Km_Per_Year",
        "Price_Ratio",
    ]
    onehot_features = ["Fuel_Type", "Selling_type", "Transmission"]
    target_enc_features = ["Car_Name"]

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_features),
            ("onehot", OneHotEncoder(handle_unknown="ignore"), onehot_features),
            (
                "brand_goodwill",
                TargetEncoder(target_type="continuous", random_state=RANDOM_STATE),
                target_enc_features,
            ),
        ]
    )
    return preprocessor


# ---------------------------------------------------------------------------
# 5. MODEL COMPARISON
# ---------------------------------------------------------------------------
def compare_models(X_train, y_train, preprocessor) -> dict:
    candidates = {
        "Linear Regression": LinearRegression(),
        "Ridge Regression": Ridge(alpha=1.0, random_state=RANDOM_STATE),
        "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE),
        "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    }

    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    results = {}

    print("\n--- 5-Fold Cross-Validation (R^2) ---")
    for name, model in candidates.items():
        pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
        scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="r2")
        results[name] = scores
        print(f"{name:20s}: mean R^2 = {scores.mean():.4f}  (+/- {scores.std():.4f})")

    return results


# ---------------------------------------------------------------------------
# 6. HYPERPARAMETER TUNING
# ---------------------------------------------------------------------------
def tune_best_model(best_name, X_train, y_train, preprocessor):
    if best_name == "Random Forest":
        model = RandomForestRegressor(random_state=RANDOM_STATE)
        param_grid = {
            "model__n_estimators": [100, 200, 400],
            "model__max_depth": [None, 6, 10, 16],
            "model__min_samples_split": [2, 5, 10],
            "model__min_samples_leaf": [1, 2, 4],
        }
    elif best_name == "Gradient Boosting":
        model = GradientBoostingRegressor(random_state=RANDOM_STATE)
        param_grid = {
            "model__n_estimators": [100, 200, 300],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__max_depth": [2, 3, 4],
        }
    elif best_name == "Ridge Regression":
        model = Ridge(random_state=RANDOM_STATE)
        param_grid = {"model__alpha": [0.1, 1.0, 5.0, 10.0, 50.0]}
    else:
        model = LinearRegression()
        param_grid = {}

    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])

    if not param_grid:
        pipe.fit(X_train, y_train)
        print(f"\n{best_name} has no hyperparameters to tune.")
        return pipe

    cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    grid = GridSearchCV(pipe, param_grid, cv=cv, scoring="r2", n_jobs=-1)
    grid.fit(X_train, y_train)

    print(f"\n--- GridSearchCV results for {best_name} ---")
    print(f"Best params: {grid.best_params_}")
    print(f"Best CV R^2: {grid.best_score_:.4f}")
    return grid.best_estimator_


# ---------------------------------------------------------------------------
# 7. FINAL EVALUATION
# ---------------------------------------------------------------------------
def evaluate_model(model, X_test, y_test) -> np.ndarray:
    y_pred = model.predict(X_test)

    r2   = r2_score(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mape = mean_absolute_percentage_error(y_test, y_pred) * 100

    print("\n=== Final Test Set Performance ===")
    print(f"R^2 Score : {r2:.4f}")
    print(f"MAE       : {mae:.3f} lakhs")
    print(f"RMSE      : {rmse:.3f} lakhs")
    print(f"MAPE      : {mape:.2f}%")

    return y_pred


def plot_results(y_test, y_pred, model, preprocessor_input_cols):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(y_test, y_pred, alpha=0.6, color="teal")
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    axes[0].plot(lims, lims, "r--", label="Perfect prediction")
    axes[0].set_xlabel("Actual Selling Price")
    axes[0].set_ylabel("Predicted Selling Price")
    axes[0].set_title("Actual vs Predicted")
    axes[0].legend()

    residuals = y_test - y_pred
    axes[1].hist(residuals, bins=25, color="indianred", edgecolor="white")
    axes[1].set_title("Residuals Distribution")
    axes[1].set_xlabel("Residual (Actual - Predicted)")

    plt.tight_layout()
    plt.savefig("model_evaluation.png", dpi=150)
    plt.close()
    print("Saved prediction/residual plots to model_evaluation.png")

    final_model = model.named_steps["model"]
    if hasattr(final_model, "feature_importances_"):
        try:
            feature_names = model.named_steps["preprocessor"].get_feature_names_out()
        except Exception:
            feature_names = preprocessor_input_cols

        importances = final_model.feature_importances_
        order = np.argsort(importances)[::-1][:15]

        plt.figure(figsize=(8, 6))
        plt.barh(range(len(order)), importances[order][::-1], color="slateblue")
        plt.yticks(range(len(order)), np.array(feature_names)[order][::-1])
        plt.xlabel("Importance")
        plt.title("Top Feature Importances")
        plt.tight_layout()
        plt.savefig("feature_importance.png", dpi=150)
        plt.close()
        print("Saved feature importance plot to feature_importance.png")


# ---------------------------------------------------------------------------
# MAIN — safe to run both as a script and inside a Jupyter notebook cell
# ---------------------------------------------------------------------------
def main():
    df = load_data(DATA_PATH)
    explore_data(df)

    df_fe = engineer_features(df)
    print("\n--- Engineered features sample ---")
    print(df_fe[["Car_Age", "Km_Per_Year", "Price_Ratio"]].head())

    X = df_fe.drop(columns=["Selling_Price"])
    y = df_fe["Selling_Price"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )
    print(f"\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

    preprocessor = build_preprocessor()

    cv_results = compare_models(X_train, y_train, preprocessor)
    best_name = max(cv_results, key=lambda k: cv_results[k].mean())
    print(f"\nBest model from cross-validation: {best_name}")

    final_model = tune_best_model(best_name, X_train, y_train, preprocessor)

    y_pred = evaluate_model(final_model, X_test, y_test)
    plot_results(y_test.values, y_pred, final_model, X.columns)

    joblib.dump(final_model, "car_price_model.pkl")
    print("\nFinal trained pipeline saved to car_price_model.pkl")

    # Demo prediction
    sample = pd.DataFrame([{
        "Car_Name": "swift",
        "Present_Price": 6.5,
        "Driven_kms": 30000,
        "Fuel_Type": "Petrol",
        "Selling_type": "Dealer",
        "Transmission": "Manual",
        "Owner": 0,
        "Car_Age": 5,
        "Km_Per_Year": 30000 / 5,
        "Price_Ratio": 6.5 / 6,
    }])
    pred_price = final_model.predict(sample)[0]
    print(f"\nDemo prediction for a sample car -> Predicted selling price: {pred_price:.2f} lakhs")


# Guard so Jupyter importing this file doesn't auto-run main()
if __name__ == "__main__":
    main()

# ---------------------------------------------------------------------------
# Jupyter convenience: call main() explicitly in the last cell of your notebook
# ---------------------------------------------------------------------------
# main()   ← uncomment this line when running inside a Jupyter notebook cell

Loaded data: 301 rows, 9 columns

--- Data types & missing values ---
Car_Name          object
Year               int64
Selling_Price    float64
Present_Price    float64
Driven_kms         int64
Fuel_Type         object
Selling_type      object
Transmission      object
Owner              int64
dtype: object

Missing values per column:
Car_Name         0
Year             0
Selling_Price    0
Present_Price    0
Driven_kms       0
Fuel_Type        0
Selling_type     0
Transmission     0
Owner            0
dtype: int64

--- Summary statistics ---
               count unique     top freq         mean           std     min  \
Car_Name         301     98    city   26          NaN           NaN     NaN   
Year           301.0    NaN     NaN  NaN  2013.627907      2.891554  2003.0   
Selling_Price  301.0    NaN     NaN  NaN     4.661296      5.082812     0.1   
Present_Price  301.0    NaN     NaN  NaN     7.628472      8.642584    0.32   
Driven_kms     301.0    NaN     NaN  NaN  36947.20598  3